# Marketing Churn: Boruta Then VIF

In [7]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from catboost_utility.boruta_catboost import BorutaCatBoost
from catboost_utility.vif_catboost import CatBoostVIF

EXAMPLES_ROOT = PROJECT_ROOT / "examples"
CATBOOST_EXPLORATION_PARAMS = {"iterations": 50, "depth": 4, "learning_rate": 0.1}

In [8]:
data_path = EXAMPLES_ROOT / "marketing_churn_data" / "marketing_churn.csv"
df = pd.read_csv(data_path)

print(f"Loaded {data_path.name} with shape {df.shape}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

Loaded marketing_churn.csv with shape (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,dtype
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


In [9]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", pd.NA), errors="coerce")

target = "Churn"
X = df.drop(columns=[target, "customerID"]).copy()
y = df[target].map({"No": 0, "Yes": 1}).astype(int)

for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
    X[col] = X[col].fillna("missing")

cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Dropped 'customerID' because it is an identifier, not a predictive signal.")
print(f"Prepared X with shape {X.shape} and target '{target}'")
print("Categorical features:", cat_features)

Dropped 'customerID' because it is an identifier, not a predictive signal.
Prepared X with shape (7043, 19) and target 'Churn'
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10928\205639556.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category", "bool"]).columns:
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10928\205639556.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydat

In [10]:
boruta = BorutaCatBoost(
    cat_features=cat_features,
    max_iter=50,
    patience=3,
    correction_method="bonferroni",
    task_type="classification",
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
boruta.fit(X, y)

boruta_selected = boruta.get_feature_names_out()
boruta_result = boruta.get_selection_result()
decision_log = boruta.decision_log_.copy()
if decision_log.empty:
    latest_boruta_decisions = decision_log
else:
    status_order = pd.CategoricalDtype(["confirmed", "tentative", "rejected"], ordered=True)
    latest_boruta_decisions = (
        decision_log.assign(status=decision_log["status"].astype(status_order))
        .sort_values(["feature", "iteration"])
        .groupby("feature", group_keys=False)
        .tail(1)
        .sort_values(["status", "feature"])
        .reset_index(drop=True)
    )

print("Boruta selected features:", boruta_selected)
print("Boruta rejected features:", boruta_result.rejected_features)
print("Boruta tentative features:", boruta_result.tentative_features)
display(latest_boruta_decisions)

if not boruta_selected:
    raise RuntimeError(
        "Boruta did not confirm any features. Increase max_iter or CatBoost iterations and rerun."
    )

Boruta selected features: ['SeniorCitizen', 'tenure', 'OnlineSecurity', 'TechSupport', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Boruta rejected features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'OnlineBackup', 'DeviceProtection', 'StreamingTV', 'StreamingMovies']
Boruta tentative features: ['MultipleLines', 'InternetService']


,iteration,iteration_seed,feature,shadow_max,hits,p_upper,p_lower,adj_p_upper,adj_p_lower,status
0,9,51,Contract,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
1,9,51,MonthlyCharges,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
2,9,51,OnlineSecurity,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
3,9,51,PaperlessBilling,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
4,9,51,PaymentMethod,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
5,14,56,SeniorCitizen,0.722776,12,0.006470,0.999084,0.032349,1.000000,confirmed
6,9,51,TechSupport,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
7,9,51,TotalCharges,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
8,9,51,tenure,1.109369,9,0.001953,1.000000,0.037109,1.000000,confirmed
9,50,92,InternetService,1.059900,31,0.059460,0.967546,0.118920,1.000000,tentative


In [11]:
X_boruta = X[boruta_selected].copy()
vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

vif = CatBoostVIF(
    cat_features=vif_cat_features,
    threshold=5.0,
    scoring_method="holdout",
    holdout_fraction=0.2,
    n_jobs=1,
    catboost_params=CATBOOST_EXPLORATION_PARAMS,
    random_state=42,
)
vif_result = vif.fit_eliminate(X_boruta)

print("VIF-retained features:", vif_result.selected_features)
print("VIF-dropped features:", vif_result.rejected_features)
display(vif_result.metrics)

elimination_history = pd.DataFrame(vif_result.config["elimination_history"])
if elimination_history.empty:
    print("No VIF eliminations were needed at the current threshold.")
else:
    display(elimination_history)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10928\1810042448.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  vif_cat_features = X_boruta.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


VIF-retained features: ['MonthlyCharges', 'tenure', 'TechSupport', 'OnlineSecurity', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen']
VIF-dropped features: ['TotalCharges']


,feature,vif,r_squared,is_categorical,clamped
0,MonthlyCharges,3.145447,0.682080,False,False
1,tenure,2.606048,0.616277,False,False
2,TechSupport,2.392638,0.582051,True,False
3,OnlineSecurity,2.350231,0.574510,True,False
4,Contract,1.571922,0.363836,True,False
5,PaymentMethod,1.168868,0.144471,True,False
6,PaperlessBilling,1.133725,0.117952,True,False
7,SeniorCitizen,1.102813,0.093228,False,False


,iteration,dropped_feature,dropped_vif,remaining_features
0,0,TotalCharges,328.999173,8


In [12]:
from catboost_utility.rfe_catboost import CatBoostRFE

rfe = CatBoostRFE(
    n_features_to_select=5,
    cat_features=vif_cat_features,
    task_type="classification",
    random_state=42,
)
rfe.fit(X_boruta, y)
rfe_selected = rfe.get_feature_names_out()
print("RFE selected features:", rfe_selected)

X_final = X_boruta[rfe_selected].copy()

print("Final feature set:", rfe_selected)
display(X_final.head())

Final feature set: ['MonthlyCharges', 'tenure', 'TechSupport', 'OnlineSecurity', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen']


,MonthlyCharges,tenure,TechSupport,OnlineSecurity,Contract,PaymentMethod,PaperlessBilling,SeniorCitizen
0,29.85,1,No,No,Month-to-month,Electronic check,Yes,0
1,56.95,34,No,Yes,One year,Mailed check,No,0
2,53.85,2,No,Yes,Month-to-month,Mailed check,Yes,0
3,42.30,45,Yes,Yes,One year,Bank transfer (automatic),No,0
4,70.70,2,No,No,Month-to-month,Electronic check,Yes,0
